# Main experiment
Recompute the deployment, size, ranking, and predictive results from one aggregate row per planned fit. Balanced accuracy (BAcc) and AUC are fractions in the CSVs.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"data/main_fit_metrics.csv").is_file())
OUT = Path(os.environ.get("ARTICLE_RESULTS_DIR", root/"generated")).resolve()
OUT.mkdir(parents=True, exist_ok=True)

def read(name):
    return pd.read_csv(root/"data"/(name+".csv"), float_precision="round_trip")

def save(name, frame):
    frame.to_csv(OUT/(name+".csv"), index=False)

def close(value, expected, tolerance):
    return abs(value-expected) <= tolerance

In [ ]:
main = read("main_fit_metrics")
keys = ["candidate", "input_shape", "encoding", "architecture"]
fit = keys + ["split_id", "seed_idx"]
val = "final_board_val_balanced_accuracy"
test = "final_board_test_balanced_accuracy"
auc = "final_board_test_roc_auc"
passed = main[main.final_strict_deployment_success]
assert len(main) == 30600 and not main.duplicated(fit).any()
assert main.groupby(keys).size().eq(60).all()
assert len(passed) == 30050
status = main.groupby("final_status").size().rename("fits").to_frame()
assert status.loc["00_original_strict_pass_305C", "fits"] == 16388
assert status.loc["01_recovered_strict_pass_305G", "fits"] == 13662
take = main.final_status.eq("01_recovered_strict_pass_305G")
assert main.loc[take, "original_failure_stage_recomputed"].eq("09_original_forward_failed_after_map").all()
status["percent_planned"] = 100 * status.fits / len(main)
save("main_deployment", status.reset_index())
describe = passed[[val, test, auc, "param_count", "fbz_size_bytes"]].describe().T
save("main_fit_distribution", describe.reset_index(names="quantity"))
assert close(passed[test].mean(), .5650, .00005)
assert close(passed[auc].mean(), .6034, .00005)
display(status, describe)

## Validation ranking and partition-level results
Rank by median validation BAcc, lower validation IQR, higher mean validation BAcc, smaller median FBZ size, then configuration name. Thresholds were selected separately within each partition and inference stage in the original code.

In [ ]:
planned = main.groupby(keys).agg(planned=("split_id", "size"), parameters=("param_count", "median"), fbz_bytes=("fbz_size_bytes", "median"))
scores = passed.groupby(keys).agg(successful=("split_id", "size"), val_median=(val, "median"), val_mean=(val, "mean"),
    val_iqr=(val, lambda x: x.quantile(.75) - x.quantile(.25)),
    test_mean=(test, "mean"), test_median=(test, "median"), test_auc_mean=(auc, "mean"))
designs = planned.join(scores).reset_index()
designs["successful"] = designs.successful.fillna(0).astype(int)
designs["configuration"] = designs[keys].agg("|".join, axis=1)
round_cols = ["val_median", "val_iqr", "val_mean"]
rounded = designs[round_cols].round(12)
designs = designs.assign(**{f"rank_{c}": rounded[c] for c in round_cols})
designs = designs.sort_values(["rank_val_median", "rank_val_iqr", "rank_val_mean", "fbz_bytes", "configuration"],
    ascending=[False, True, False, True, True]).drop(columns=[f"rank_{c}" for c in round_cols])
designs.insert(0, "validation_rank", range(1, len(designs)+1))
assert len(designs) == 510 and designs.successful.gt(0).sum() == 504
save("main_configurations", designs)
shortlist = designs.head(10).copy()
shortlist["model"] = "M" + shortlist.validation_rank.astype(str)
assert shortlist.successful.eq(60).all()
top = passed.merge(shortlist[keys+["model", "validation_rank"]], on=keys, validate="many_to_one")
partition_means = top.groupby(["model", "split_id"])[[val, test, auc]].mean().reset_index()
assert partition_means.groupby("model").size().eq(20).all()
save("main_shortlist_partitions", partition_means)
groups = shortlist.assign(group=np.where(shortlist.validation_rank.le(5), "1–5", "6–10")).groupby("group").agg(
    fits=("successful", "sum"), test_bacc=("test_mean", "mean"), test_auc=("test_auc_mean", "mean"),
    median_parameters=("parameters", "median"), median_fbz_bytes=("fbz_bytes", "median"))
assert groups.fits.tolist() == [300, 300]
save("main_rank_groups", groups.reset_index())
m9 = top[top.model.eq("M9")]
m9_partitions = partition_means[partition_means.model.eq("M9")]
m9_values = m9_partitions[auc]
assert len(m9) == 60 and close(m9[auc].mean(), .6710, .00005)
assert close(m9[test].mean(), .6063, .00005)
assert shortlist.loc[shortlist.model.eq("M9"), "parameters"].item() == 1712
assert shortlist.loc[shortlist.model.eq("M9"), "fbz_bytes"].item() == 3812
assert round(m9_values.min(), 3) == .587 and round(m9_values.max(), 3) == .803
assert m9_values.ge(.70).sum() == 6
two = m9_partitions.nlargest(2, test)
assert set(two.split_id) == {2, 8}
assert close(two[test].mean(), .7238, .00005) and close(two[auc].mean(), .7951, .00005)
display(shortlist, groups, m9_partitions, two)

## Matched stages and design contrasts
The stage comparison uses the same 30,050 successful fits. Design contrasts pair fits by all other design keys, partition and seed.

In [ ]:
stage_names = {"float": "Floating-point", "akida_predict": "Akida software", "final_board": "Physical board"}
stages = pd.DataFrame([
    {"stage": name, "fits": len(passed), "val_bacc": passed[f"{prefix}_val_balanced_accuracy"].mean(),
     "test_bacc": passed[f"{prefix}_test_balanced_accuracy"].mean(), "test_auc": passed[f"{prefix}_test_roc_auc"].mean()}
    for prefix, name in stage_names.items()
])
save("main_stage_summary", stages)
change = passed[test] - passed.akida_predict_test_balanced_accuracy
assert change.abs().lt(1e-12).sum() == 21544
assert change.abs().ge(.05-1e-12).sum() == 1341
paired = pd.DataFrame([{"comparison": f"{a} → {b}", "mean_bacc_change": ba.mean(), "median_bacc_change": ba.median(),
    "equal_bacc": ba.abs().lt(1e-12).sum(), "bacc_change_at_least_5pp": ba.abs().ge(.05-1e-12).sum(), "mean_auc_change": au.mean()}
    for a, b in zip(stage_names, list(stage_names)[1:])
    for ba, au in [(passed[f"{b}_test_balanced_accuracy"]-passed[f"{a}_test_balanced_accuracy"],
                    passed[f"{b}_test_roc_auc"]-passed[f"{a}_test_roc_auc"])]])
save("main_stage_deltas", paired)

def contrast(name, column, left, right, frame=main):
    paired_on = [k for k in fit if k != column]
    cols = paired_on + ["final_strict_deployment_success", test, auc]
    matched = frame.loc[frame[column].eq(left), cols].merge(frame.loc[frame[column].eq(right), cols],
        on=paired_on, suffixes=("_left", "_right"), validate="one_to_one")
    valid = matched[matched.final_strict_deployment_success_left & matched.final_strict_deployment_success_right].copy()
    valid["delta"] = valid[test+"_right"] - valid[test+"_left"]
    by_split = valid.groupby("split_id").delta.mean()
    by_context = valid.groupby([k for k in keys if k != column]).delta.mean()
    return {"contrast": name, "eligible": len(matched), "both_passed": len(valid), "test_bacc_delta": valid.delta.mean(),
        "test_auc_delta": (valid[auc+"_right"]-valid[auc+"_left"]).mean(), "min_split_delta": by_split.min(),
        "max_split_delta": by_split.max(), "positive_contexts": by_context.gt(0).sum(), "positive_partitions": by_split.gt(0).sum()}

contrasts = pd.DataFrame([
    contrast("C4 to C8", "architecture", "singleconv4_dense2", "singleconv8_dense2"),
    contrast("C8 to C16", "architecture", "singleconv8_dense2", "singleconv16_dense2"),
    contrast("C4 to C16", "architecture", "singleconv4_dense2", "singleconv16_dense2"),
    contrast("C16 add H8", "architecture", "singleconv16_dense2", "singleconv16_dense8_dense2"),
    contrast("Two C16 mixed to same", "architecture", "conv16_conv16_dense2_first_valid_rest_same", "conv16_conv16_dense2_all_same"),
    contrast("Complete groups fixed to ranked", "encoding", "fixed_order", "train_ranked_order", main[main.candidate.isin(["timing", "transcript_timing"])]),
    contrast("9x9 to 11x11", "input_shape", "9x9x1", "11x11x1"),
    contrast("Transcript and timing 25 to 47", "candidate", "transcript_timing_top25", "transcript_timing_top47"),
    contrast("Transcript and timing 47 to all 81", "candidate", "transcript_timing_top47", "all_features_top81"),
])
assert contrasts.both_passed.tolist() == [2040,2040,2040,2034,2040,6176,8859,6989,3544]
assert close(contrasts.iloc[2].test_bacc_delta, .0339, .00005)
save("main_matched_contrasts", contrasts)
baseline = read("matched_baseline_metrics")
board = top.groupby(["model", "candidate", "split_id"])[test].mean().reset_index()
baseline_join = board.merge(baseline, on=["candidate","split_id"], suffixes=("_board","_baseline"))
baseline_summary = baseline_join.groupby(["model_board","model_baseline"]).agg(splits=("split_id","size"),
    board_test_bacc=(test,"mean"), baseline_test_bacc=("test_balanced_accuracy","mean")).reset_index()
assert baseline_summary.splits.eq(20).all()
save("main_matched_baselines", baseline_summary)
display(stages, paired, contrasts, baseline_summary)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.1, 4))
for metric, label in [(val, "Validation"), (test, "Test")]:
    bins = np.arange(10, 103, 3)
    ax.hist(100*passed[metric], bins, weights=np.repeat(100/len(passed), len(passed)), histtype="step", label=label)
ax.set(xlabel="Physical-board balanced accuracy (%)", ylabel="Fits in bin (%)")
ax.legend()
fig.savefig(OUT/"main_fit_distribution.png", dpi=300, bbox_inches="tight")
plt.close(fig)

fig, axes = plt.subplots(2, 1, figsize=(7.1, 6.5))
good = designs[designs.successful.gt(0)]
for ax, x, y, ylabel in [(axes[0], "parameters", "test_mean", "Mean test BAcc"), (axes[1], "fbz_bytes", "test_auc_mean", "Mean test AUC")]:
    for feature, rows in good.groupby("candidate"):
        ax.scatter(rows[x], rows[y], s=10, alpha=.65, label=feature)
    ax.set(xscale="log", xlabel=x.replace("_", " "), ylabel=ylabel)
axes[1].legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(OUT/"main_compactness.png", dpi=300, bbox_inches="tight")
plt.close(fig)